# 建模（資料科學家）

讀 `input/cleaned.csv`，寫出 `output/model_report.json`。

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

Path("output").mkdir(exist_ok=True)
df = pd.read_csv("input/cleaned.csv")
# 不用 AGE、MARITAL_STATUS（法遵受限），不用 LEAK_FUTURE_DEFAULT（目標洩漏）
FEATURES = ["PAY_0", "BILL_AMT1"]
X, y = df[FEATURES], df["default"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(len(X_train), len(X_val))

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
gap = round(train_auc - val_auc, 4)
print("train AUC", round(train_auc, 4), "val AUC", round(val_auc, 4), "gap", gap)

In [ ]:
coef = np.abs(model.named_steps["logisticregression"].coef_[0])
importances = {f: round(float(c / coef.sum()), 4) for f, c in zip(FEATURES, coef)}
report = {
    "features": FEATURES,
    "model_type": "logistic_regression",
    "train_auc": round(float(train_auc), 4),
    "val_auc": round(float(val_auc), 4),
    "overfit_gap": gap,
    "diagnosis": "健康：落差小於 0.05，沒有過擬合" if abs(gap) < 0.05 else "需注意：訓練與驗證落差偏大",
    "feature_importances": importances,
    "rows_train": len(X_train),
    "rows_val": len(X_val),
}
Path("output/model_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
report

## 特徵決策理由

只用 PAY_0 和 BILL_AMT1。PAY_0 是 BA 已經證明跟違約單調相關的欄位；BILL_AMT1 代表曝險金額，業務上合理。排除 AGE、MARITAL_STATUS（法遵禁用於授信），排除 LEAK_FUTURE_DEFAULT（它是事後才知道的催收結果，用了會讓離線 AUC 虛高、上線失效）。

## 模型風險

邏輯斯迴歸，train AUC 0.785、val AUC 0.783，落差 0.002，沒有過擬合。風險有三：
1. PAY_0 權重占 88%，模型幾乎只看還款延遲；對「第一次遲繳」的新客戶判斷力有限。
2. PAY_0 有 3% 是補值，這些人的分數偏低。
3. BILL_AMT1 可能間接反映年齡或收入，是受限特徵的代理變數風險，需要 QA 留意。

## 給下游的提醒

MLE：模型只有兩個特徵加標準化，延遲不會是問題，請照 `features` 重建。Visual：係數已標準化，可以直接比大小；對客戶解釋時請只講 PAY_0，BILL_AMT1 的貢獻很小。QA：Client 可能問「為什麼不用年齡」，答案是法遵，不是效能。